# League of Legends object detection: remaining experiments in one run

**What this notebook does**, all in one "Run all":

1. **Image-size experiment:** trains YOLOv8n, YOLO11n and YOLO26n at 1280 px instead of 640 px, to test whether small objects (minions) are detected better when the screenshot is shrunk less.
2. **Baselines:** trains Faster R-CNN and EfficientDet-D0 with the same epoch and early-stopping settings as the main YOLO runs (30 / 30).
3. **Master table:** collects every summary table from every notebook (April runs, all YOLO experiments, baselines) into one file for the report.

Everything it creates goes into its own folder, `runs_v3`. It only *reads* from `runs_v2` (the results of `lol_detection_compare-v2`) to build the master table, and never changes anything there.

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all.
If Colab disconnects, run all again. Finished models are skipped and a half-trained one resumes.

**Why two different kinds of model?** The YOLO models are close cousins. Faster R-CNN is a *two-stage* detector: stage one says "something might be here", stage two looks closer and decides what it is, like a teacher who first circles suspicious answers and then checks each one. EfficientDet is a one-stage detector like YOLO but built differently (EfficientNet backbone, BiFPN). Adding them makes the comparison about *approaches*, not only *versions*.

## 1. Setup

In [1]:
# effdet is the library that contains EfficientDet. ultralytics is only used here for its metric code,
# so that every model in the report is scored with the same formula.
!pip install -q effdet ultralytics

from google.colab import drive
drive.mount("/content/drive")

import torch
print("GPU available:", torch.cuda.is_available())

# Safety stop: training on the CPU works but takes many hours, so we refuse to start by accident.
ALLOW_CPU = False
assert torch.cuda.is_available() or ALLOW_CPU, (
    "No GPU found. Go to Runtime > Change runtime type, choose a GPU (for example T4), then Run all again."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU available: True


## 2. Settings

These mirror the YOLO notebook wherever the two kinds of model share a setting (epochs, patience, image size, seed). Two things cannot be identical, and the report should say so openly:

- **Learning recipe:** YOLO comes with its own tuned optimiser and heavy augmentation (mosaic and more). The models here use one plain, standard recipe (AdamW, horizontal flip only).
- **Batch size:** Faster R-CNN is about ten times bigger than a nano YOLO, so it uses a smaller batch to fit in GPU memory.

In [2]:
from pathlib import Path

# ---- Switch parts on or off ----
RUN_YOLO_EXPERIMENT = True   # Part 1: YOLO image-size experiment
RUN_BASELINES       = True   # Part 2: Faster R-CNN and EfficientDet

# ---- Shared settings, matching the main YOLO results (e30_p30_img640) ----
EPOCHS   = 30
PATIENCE = 30
SEED     = 0

# ---- Part 1: YOLO image-size experiment ----
YOLO_MODELS = [                # all three, so the image-size effect can be compared across models
    "yolov8n.pt",
    "yolo11n.pt",
    "yolo26n.pt",
]
YOLO_IMGSZ  = 1280             # the main runs used 640
YOLO_BATCH  = 16               # same as the main runs. If the T4 runs out of memory, use 8 and say so in the report.

# ---- Part 2: baselines ----
BASELINES = [
    "fasterrcnn_resnet50_fpn_v2",
    "efficientdet_d0",
]
IMGSZ      = 640      # longest image side, same as the main YOLO runs
BATCH      = 8        # smaller than YOLO's 16 because Faster R-CNN needs more memory
PRETRAINED = True     # start from weights that already learned general shapes on the COCO dataset
# Learning rate: how big each correction step is. Common fine-tuning starting points, not tuned values.
LR = {
    "fasterrcnn_resnet50_fpn_v2": 1e-4,
    "efficientdet_d0": 5e-4,
}

CLASS_NAMES = ["ally_minion", "enemy_minion", "enemy_champion", "ally_champion", "tower", "player"]
EXPECTED = {"train": 661, "val": 94, "test": 79}

DRIVE_PROJECT = None  # leave as None to auto-detect, or set a Path by hand

In [3]:
MYDRIVE = Path("/content/drive/MyDrive")

def find_project() -> Path:
    for pattern in ("computer_vision", "*/computer_vision", "*/*/computer_vision", "*/*/*/computer_vision"):
        for candidate in MYDRIVE.glob(pattern):
            if (candidate / "screenshot_generator" / "dataset" / "images" / "train").exists():
                return candidate
    raise FileNotFoundError("Could not find computer_vision. Set DRIVE_PROJECT by hand in the settings cell.")

if DRIVE_PROJECT is None:
    DRIVE_PROJECT = find_project()

DRIVE_DATASET = DRIVE_PROJECT / "screenshot_generator" / "dataset"
NEW_RUNS_DIR  = DRIVE_PROJECT / "runs_v3"                 # everything this notebook creates goes here
SUMMARY_DIR   = NEW_RUNS_DIR / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
V2_SUMMARY_DIR = DRIVE_PROJECT / "runs_v2" / "summary"   # results of lol_detection_compare-v2 (read only)
LOCAL_DATASET = Path("/content/dataset")
# Short labels for the settings. They go into run names and summary file names, so nothing gets overwritten.
SETTINGS_TAG = f"e{EPOCHS}_p{PATIENCE}_img{IMGSZ}"        # baselines
YOLO_TAG     = f"e{EPOCHS}_p{PATIENCE}_img{YOLO_IMGSZ}"   # YOLO image-size experiment
print("Project :", DRIVE_PROJECT)
print("Baselines settings:", SETTINGS_TAG)
print("YOLO experiment settings:", YOLO_TAG)

Project : /content/drive/MyDrive/Projects/Uni Stuff/computer_vision
Baselines settings: e30_p30_img640
YOLO experiment settings: e30_p30_img1280


## 3. Copy the dataset to the local disk (skipped if it is already there)

In [4]:
import shutil

IMAGE_TYPES = {".png", ".jpg", ".jpeg"}

def count_images(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(1 for f in folder.iterdir() if f.suffix.lower() in IMAGE_TYPES)

for split in EXPECTED:
    for kind in ("images", "labels"):
        src = DRIVE_DATASET / kind / split
        dst = LOCAL_DATASET / kind / split
        if kind == "images" and count_images(dst) == EXPECTED[split]:
            continue
        print(f"Copying {kind}/{split} ...")
        shutil.copytree(src, dst, dirs_exist_ok=True)

for split, expected in EXPECTED.items():
    n_img = count_images(LOCAL_DATASET / "images" / split)
    print(f"{split:5s}: {n_img} images (expected {expected})")
    assert n_img == expected, f"{split} has {n_img} images but {expected} were expected."

Copying labels/train ...
Copying labels/val ...
Copying labels/test ...
train: 661 images (expected 661)
val  : 94 images (expected 94)
test : 79 images (expected 79)


## Part 1: YOLO image-size experiment

The screenshots are 1920 px wide. At 640 px, everything is shrunk to a third of its size, so a minion that was 45 px wide becomes about 15 px. That is like trying to read a car number plate from the far end of a car park. At 1280 px, things are shrunk only to two thirds.

**What can go wrong:**
- 1280 px images need about four times the memory. If you see `CUDA out of memory`, set `YOLO_BATCH = 8` in the settings and run all again.
- Training and scoring take roughly four times longer than at 640 px.
- Only one setting changes (image size), so any difference in the minion scores can be traced back to it.
- Three models at 1280 px take a while. If time is short, remove lines from `YOLO_MODELS`; finished models are kept either way.

In [5]:
import gc
import yaml
import pandas as pd
from ultralytics import YOLO

# The dataset "address card" for YOLO. Written fresh because the old dataset.yaml points to a folder on your Mac.
DATA_YAML = LOCAL_DATASET / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(LOCAL_DATASET),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}, sort_keys=False))

def yolo_run_name(model_file: str) -> str:
    return f"detect_{Path(model_file).stem}_{YOLO_TAG}"

def yolo_evaluate(weights: Path, label: str, split: str, save_name: str):
    # Same scoring code as the compare notebook, but with the experiment's image size.
    model = YOLO(str(weights))
    metrics = model.val(data=str(DATA_YAML), split=split, imgsz=YOLO_IMGSZ, batch=YOLO_BATCH,
                        project=str(NEW_RUNS_DIR / "eval"), name=save_name, exist_ok=True,
                        plots=True, verbose=False)
    box = metrics.box
    overall = {
        "model": label, "split": split, "settings": YOLO_TAG,
        "precision": round(float(box.mp), 3), "recall": round(float(box.mr), 3),
        "mAP50": round(float(box.map50), 3), "mAP50-95": round(float(box.map), 3),
        "inference_ms_per_image": round(float(metrics.speed["inference"]), 2),
        "params_millions": round(sum(p.numel() for p in model.model.parameters()) / 1e6, 2),
    }
    per_class = [{
        "model": label, "split": split, "settings": YOLO_TAG, "class": metrics.names[int(c)],
        "precision": round(float(box.p[i]), 3), "recall": round(float(box.r[i]), 3),
        "AP50": round(float(box.ap50[i]), 3), "AP50-95": round(float(box.ap[i]), 3),
    } for i, c in enumerate(box.ap_class_index)]
    return overall, per_class

In [6]:
if RUN_YOLO_EXPERIMENT:
    device = 0 if torch.cuda.is_available() else "cpu"
    for model_file in YOLO_MODELS:
        name = yolo_run_name(model_file)
        run_dir = NEW_RUNS_DIR / name
        done_marker = run_dir / "DONE.txt"
        last_checkpoint = run_dir / "weights" / "last.pt"
        print("=" * 70)
        if done_marker.exists():
            print(f"{name}: already finished, skipping.")
            continue
        if last_checkpoint.exists():
            print(f"{name}: found a checkpoint, resuming.")
            try:
                YOLO(str(last_checkpoint)).train(resume=True, device=device)
            except AssertionError as err:
                print(f"Nothing left to resume ({err}). Marking as finished.")
        else:
            print(f"{name}: starting training.")
            YOLO(model_file).train(data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=YOLO_IMGSZ,
                                   batch=YOLO_BATCH, seed=SEED, device=device, project=str(NEW_RUNS_DIR),
                                   name=name, exist_ok=True, plots=True)
        done_marker.write_text("finished")

    # Score on val and test
    yolo_overall, yolo_per_class = [], []
    for model_file in YOLO_MODELS:
        name = yolo_run_name(model_file)
        run_dir = NEW_RUNS_DIR / name
        history = pd.read_csv(run_dir / "results.csv")
        history.columns = [c.strip() for c in history.columns]
        for split in ("val", "test"):
            overall, per_class = yolo_evaluate(run_dir / "weights" / "best.pt", Path(model_file).stem, split,
                                               save_name=f"{name}_{split}")
            overall["epochs_run"] = len(history)
            overall["best_epoch"] = int(history.loc[history["metrics/mAP50-95(B)"].idxmax(), "epoch"])
            yolo_overall.append(overall)
            yolo_per_class.extend(per_class)

    pd.DataFrame(yolo_overall).to_csv(SUMMARY_DIR / f"new_models_overall_{YOLO_TAG}.csv", index=False)
    pd.DataFrame(yolo_per_class).to_csv(SUMMARY_DIR / f"new_models_per_class_{YOLO_TAG}.csv", index=False)
    display(pd.DataFrame(yolo_overall))

    gc.collect()
    torch.cuda.empty_cache()   # hand the GPU memory back before the baselines start
else:
    print("Part 1 switched off.")

detect_yolov8n_e30_p30_img1280: already finished, skipping.
detect_yolo11n_e30_p30_img1280: already finished, skipping.
detect_yolo26n_e30_p30_img1280: already finished, skipping.
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,922 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 220.9±80.5 MB/s, size: 2178.4 KB)
val: Scanning /content/dataset/labels/val.cache... 94 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 94/94 14.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.2s/it 13.5s
                   all         94        431      0.883      0.911      0.937      0.711
Speed: 7.5ms preprocess, 10.6ms inference, 0.0ms loss, 3.9ms postprocess per image
Results saved to /content/drive/MyDrive/Projects/Uni Stuff/computer_vision/runs_v3/eval/detect_yolov8n_e30_p30_img1280_val
Ultralytics 8.4.215 🚀 Py

,model,split,settings,precision,recall,mAP50,mAP50-95,inference_ms_per_image,params_millions,epochs_run,best_epoch
0,yolov8n,val,e30_p30_img1280,0.883,0.911,0.937,0.711,10.62,3.14,30,28
1,yolov8n,test,e30_p30_img1280,0.885,0.858,0.907,0.675,9.84,3.14,30,28
2,yolo11n,val,e30_p30_img1280,0.916,0.897,0.943,0.714,9.67,2.61,30,25
3,yolo11n,test,e30_p30_img1280,0.861,0.882,0.912,0.683,10.25,2.61,30,25
4,yolo26n,val,e30_p30_img1280,0.911,0.908,0.938,0.710,10.06,2.54,30,24
5,yolo26n,test,e30_p30_img1280,0.888,0.867,0.916,0.699,9.71,2.54,30,24


## Part 2: Faster R-CNN and EfficientDet-D0

Two things cannot be identical to the YOLO runs, and the report should say so openly:

- **Learning recipe:** YOLO comes with its own tuned optimiser and heavy augmentation (mosaic and more). These models use one plain, standard recipe (AdamW, horizontal flip only).
- **Batch size:** Faster R-CNN is about ten times bigger than a nano YOLO, so it uses a smaller batch to fit in GPU memory.

Like YOLO, the learning rate here slows down over the full 30 epochs, so the schedule matches the main YOLO runs.

### Loading the images

YOLO label files describe a box as "centre and size, as fractions of the picture". The models here want "left, top, right, bottom, in pixels". This class does that translation.

It also shrinks every screenshot from 1920x1080 to 640x360 and adds a thin grey strip at the bottom to reach 640x384. Why 384? EfficientDet halves the picture seven times inside itself, so both sides must be divisible by 128. It is like cutting a cake into equal slices: that only works cleanly if the cake size divides evenly.

All images are loaded into memory once. Reading 2 MB screenshots from disk on every epoch would be the slowest part of training otherwise.

In [7]:
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

IMG_W = IMGSZ                                # 640
IMG_H = int(np.ceil(IMGSZ * 9 / 16 / 128) * 128)   # 360 rounded up to a multiple of 128 = 384

class YoloBoxDataset(Dataset):
    def __init__(self, split: str, augment: bool):
        self.augment = augment
        self.items = []
        image_dir = LOCAL_DATASET / "images" / split
        label_dir = LOCAL_DATASET / "labels" / split
        paths = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_TYPES)
        for path in paths:
            image = Image.open(path).convert("RGB")
            w0, h0 = image.size
            scale = min(IMG_W / w0, IMG_H / h0)
            new_w, new_h = round(w0 * scale), round(h0 * scale)
            small = np.asarray(image.resize((new_w, new_h), Image.BILINEAR))

            boxes, labels = [], []
            label_file = label_dir / (path.stem + ".txt")
            if label_file.exists():
                for line in label_file.read_text().splitlines():
                    parts = line.split()
                    if len(parts) != 5:
                        continue
                    cls, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
                    x1, y1 = (xc - w / 2) * new_w, (yc - h / 2) * new_h
                    x2, y2 = (xc + w / 2) * new_w, (yc + h / 2) * new_h
                    x1, y1, x2, y2 = max(x1, 0), max(y1, 0), min(x2, new_w), min(y2, new_h)
                    # Boxes thinner than one pixel after shrinking would crash Faster R-CNN, so they are dropped.
                    if x2 - x1 >= 1 and y2 - y1 >= 1:
                        boxes.append([x1, y1, x2, y2])
                        labels.append(cls)
            self.items.append((small, np.array(boxes, dtype=np.float32).reshape(-1, 4), np.array(labels, dtype=np.int64)))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        small, boxes, labels = self.items[index]
        boxes = boxes.copy()
        new_h, new_w = small.shape[:2]
        if self.augment and random.random() < 0.5:
            # Mirror the picture left to right, and mirror the boxes with it.
            small = small[:, ::-1]
            boxes[:, [0, 2]] = new_w - boxes[:, [2, 0]]
        canvas = np.full((IMG_H, IMG_W, 3), 114, dtype=np.uint8)   # grey background
        canvas[:new_h, :new_w] = small
        image = torch.from_numpy(canvas).permute(2, 0, 1).float() / 255.0
        return image, torch.from_numpy(boxes), torch.from_numpy(labels)

def collate(batch):
    # Images all have the same size so they can be stacked. Boxes stay as a list because each image has a different number.
    images, boxes, labels = zip(*batch)
    return torch.stack(images), list(boxes), list(labels)

datasets = {}
if RUN_BASELINES:
    print(f"Model input size: {IMG_W}x{IMG_H}")
    datasets = {split: YoloBoxDataset(split, augment=(split == "train")) for split in EXPECTED}
    for split, ds in datasets.items():
        print(f"{split:5s}: {len(ds)} images, {sum(len(b) for _, b, _ in ds.items)} boxes")

Model input size: 640x384
train: 661 images, 3058 boxes
val  : 94 images, 431 boxes
test : 79 images, 411 boxes


### The two models, behind one shared interface

The two libraries speak different "dialects" (different box orders, different class numbering). Each wrapper below hides that, so the training loop only ever needs two questions:

- `loss(...)`: "how wrong are you on these training images?"
- `predict(...)`: "what do you see in these images?"

**What can go wrong here:** class numbering. Your labels count from 0. Both libraries keep a special number for "background", so internally the classes are shifted by one and shifted back on the way out. Getting that wrong gives a model that trains fine but scores zero.

In [8]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from effdet import create_model, DetBenchPredict

NUM_CLASSES = len(CLASS_NAMES)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class FasterRCNNWrapper:
    def __init__(self):
        self.model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
            weights="DEFAULT" if PRETRAINED else None,
            weights_backbone=None,              # only matters when weights is None
            min_size=IMG_H, max_size=IMG_W,     # keep our 640x384 input as it is
            box_score_thresh=0.001,             # keep low-confidence boxes; the metric code needs them
            box_detections_per_img=300,
        )
        # Swap the final layer: the pretrained one knows 91 COCO classes, we need 6 (+1 for background).
        in_features = self.model.roi_heads.box_predictor.cls_score.in_features
        self.model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES + 1)
        self.model.to(DEVICE)

    def loss(self, images, boxes, labels):
        targets = [{"boxes": b.to(DEVICE), "labels": (l + 1).to(DEVICE)} for b, l in zip(boxes, labels)]
        losses = self.model(list(images.to(DEVICE)), targets)   # a dictionary of four partial losses
        return sum(losses.values())

    @torch.no_grad()
    def predict(self, images):
        outputs = self.model(list(images.to(DEVICE)))
        return [(o["boxes"], o["scores"], o["labels"] - 1) for o in outputs]

class EfficientDetWrapper:
    def __init__(self):
        self.bench = create_model(
            "tf_efficientdet_d0",
            bench_task="train",
            bench_labeler=True,                # let effdet match boxes to its anchors for us
            num_classes=NUM_CLASSES,
            pretrained=PRETRAINED,
            pretrained_backbone=PRETRAINED,
            image_size=(IMG_H, IMG_W),
            max_det_per_image=300,
        ).to(DEVICE)
        self.model = self.bench                      # what gets trained and saved
        self.predictor = DetBenchPredict(self.bench.model).to(DEVICE)   # shares the same weights

    def _normalise(self, images):
        return (images.to(DEVICE) - MEAN.to(DEVICE)) / STD.to(DEVICE)

    def loss(self, images, boxes, labels):
        target = {
            "bbox": [b[:, [1, 0, 3, 2]].to(DEVICE) for b in boxes],   # effdet wants (top, left, bottom, right)
            "cls":  [(l + 1).float().to(DEVICE) for l in labels],
        }
        return self.bench(self._normalise(images), target)["loss"]

    @torch.no_grad()
    def predict(self, images):
        detections = self.predictor(self._normalise(images))   # (batch, 300, 6): x1, y1, x2, y2, score, class
        results = []
        for det in detections:
            det = det[det[:, 4] > 0.001]
            results.append((det[:, :4], det[:, 4], det[:, 5].long() - 1))
        return results

    def train(self):
        self.bench.train()

WRAPPERS = {
    "fasterrcnn_resnet50_fpn_v2": FasterRCNNWrapper,
    "efficientdet_d0": EfficientDetWrapper,
}

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


### Scoring with the same formula as YOLO

If two teachers mark the same exam with different marking schemes, the grades cannot be compared. So this notebook borrows Ultralytics' own marking function (`ap_per_class`) and its matching rule. Every model in your report is then graded the same way.

The matching rule, in plain words: a predicted box counts as correct if it has the right class and overlaps a real box enough, and each real box can only be "claimed" by one prediction (the best-overlapping one).

In [9]:
import time
from torchvision.ops import box_iou
from ultralytics.utils.metrics import ap_per_class

IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)   # mAP50-95 averages over these ten strictness levels

def match_predictions(pred_boxes, pred_cls, gt_boxes, gt_cls):
    # Returns a True/False table: one row per prediction, one column per IoU threshold.
    correct = np.zeros((len(pred_boxes), len(IOU_THRESHOLDS)), dtype=bool)
    if len(pred_boxes) == 0 or len(gt_boxes) == 0:
        return correct
    iou = box_iou(gt_boxes, pred_boxes)                                # overlap of every real box with every prediction
    iou = (iou * (gt_cls[:, None] == pred_cls[None, :])).cpu().numpy()  # overlap only counts if the class matches
    for i, threshold in enumerate(IOU_THRESHOLDS):
        matches = np.array(np.nonzero(iou >= threshold)).T             # pairs of (real box, prediction)
        if len(matches):
            if len(matches) > 1:
                matches = matches[iou[matches[:, 0], matches[:, 1]].argsort()[::-1]]   # best overlaps first
                matches = matches[np.unique(matches[:, 1], return_index=True)[1]]      # one real box per prediction
                matches = matches[np.unique(matches[:, 0], return_index=True)[1]]      # one prediction per real box
            correct[matches[:, 1], i] = True
    return correct

def score(all_correct, all_conf, all_pred_cls, all_gt_cls):
    # Turns the collected matches into precision, recall and mAP, overall and per class.
    names = dict(enumerate(CLASS_NAMES))
    result = ap_per_class(
        np.concatenate(all_correct), np.concatenate(all_conf),
        np.concatenate(all_pred_cls), np.concatenate(all_gt_cls), names=names,
    )
    p, r, ap, classes = result[2], result[3], result[5], result[6]
    overall = {
        "precision": round(float(p.mean()), 3),
        "recall": round(float(r.mean()), 3),
        "mAP50": round(float(ap[:, 0].mean()), 3),
        "mAP50-95": round(float(ap.mean()), 3),
    }
    per_class = [
        {"class": names[int(c)], "precision": round(float(p[i]), 3), "recall": round(float(r[i]), 3),
         "AP50": round(float(ap[i, 0]), 3), "AP50-95": round(float(ap[i].mean()), 3)}
        for i, c in enumerate(classes)
    ]
    return overall, per_class

def evaluate(wrapper, split: str):
    wrapper.model.eval()
    loader = DataLoader(datasets[split], batch_size=BATCH, shuffle=False, collate_fn=collate)
    all_correct, all_conf, all_pred_cls, all_gt_cls = [], [], [], []
    seconds, timed_images = 0.0, 0
    for batch_index, (images, boxes, labels) in enumerate(loader):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()        # make sure the stopwatch measures finished work
        start = time.perf_counter()
        predictions = wrapper.predict(images)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        if batch_index > 0:                 # the first batch is a slow warm-up, so it is not timed
            seconds += time.perf_counter() - start
            timed_images += len(images)
        for (p_boxes, p_scores, p_cls), g_boxes, g_cls in zip(predictions, boxes, labels):
            order = p_scores.argsort(descending=True)
            p_boxes, p_scores, p_cls = p_boxes[order].cpu(), p_scores[order].cpu(), p_cls[order].cpu()
            all_correct.append(match_predictions(p_boxes, p_cls, g_boxes, g_cls))
            all_conf.append(p_scores.numpy())
            all_pred_cls.append(p_cls.numpy())
            all_gt_cls.append(g_cls.numpy())
    overall, per_class = score(all_correct, all_conf, all_pred_cls, all_gt_cls)
    overall["inference_ms_per_image"] = round(1000 * seconds / max(timed_images, 1), 2)
    return overall, per_class

### The training loop

One round ("epoch") goes like this: show the model every training image, let it correct itself a little after each batch, then give it the validation images as a quiz. If the quiz score is the best so far, save the model. If the score has not improved for `PATIENCE` rounds, stop.

Small safety features and why they are there:

- **Warm-up:** the first three epochs use smaller correction steps, like stretching before a sprint. Big steps on a fresh model can throw it off balance.
- **Gradient clipping:** caps any single correction, so one strange batch cannot wreck the model.
- **Mixed precision (`autocast`):** does some maths with shorter numbers, which is faster and uses less GPU memory and electricity.
- **`last.pt` on Drive after every epoch:** lets a disconnected session carry on.

In [10]:
import math
import pandas as pd

def run_name(key: str) -> str:
    return f"detect_{key}_{SETTINGS_TAG}"

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def train_baseline(key: str):
    run_dir = NEW_RUNS_DIR / run_name(key)
    weights_dir = run_dir / "weights"
    weights_dir.mkdir(parents=True, exist_ok=True)
    done_marker, last_path, best_path = run_dir / "DONE.txt", weights_dir / "last.pt", weights_dir / "best.pt"

    if done_marker.exists():
        print(f"{run_name(key)}: already finished, skipping.")
        return

    set_seed(SEED)
    wrapper = WRAPPERS[key]()
    optimizer = torch.optim.AdamW([p for p in wrapper.model.parameters() if p.requires_grad], lr=LR[key], weight_decay=1e-4)

    def lr_factor(epoch):   # 3 warm-up epochs, then a smooth slide down to 1% of the starting value
        if epoch < 3:
            return (epoch + 1) / 3
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (epoch - 3) / max(EPOCHS - 3, 1)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)

    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    start_epoch, best_score, epochs_without_improvement, history = 0, -1.0, 0, []

    if last_path.exists():
        checkpoint = torch.load(last_path, map_location=DEVICE, weights_only=False)
        wrapper.model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        start_epoch, best_score = checkpoint["epoch"] + 1, checkpoint["best_score"]
        epochs_without_improvement, history = checkpoint["epochs_without_improvement"], checkpoint["history"]
        print(f"{run_name(key)}: resuming from epoch {start_epoch + 1}.")
    else:
        print(f"{run_name(key)}: starting training.")

    loader = DataLoader(datasets["train"], batch_size=BATCH, shuffle=True, collate_fn=collate, drop_last=True)

    for epoch in range(start_epoch, EPOCHS):
        if epochs_without_improvement >= PATIENCE:
            break
        wrapper.model.train()
        running_loss, started = 0.0, time.time()
        for images, boxes, labels in loader:
            optimizer.zero_grad()
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                loss = wrapper.loss(images, boxes, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(wrapper.model.parameters(), max_norm=10.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
        scheduler.step()

        overall, _ = evaluate(wrapper, "val")
        row = {"epoch": epoch + 1, "train_loss": round(running_loss / len(loader), 4), **overall,
               "seconds": round(time.time() - started, 1)}
        history.append(row)
        print(row)

        if overall["mAP50-95"] > best_score:
            best_score, epochs_without_improvement = overall["mAP50-95"], 0
            torch.save(wrapper.model.state_dict(), best_path)
        else:
            epochs_without_improvement += 1

        torch.save({"model": wrapper.model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "epoch": epoch, "best_score": best_score,
                    "epochs_without_improvement": epochs_without_improvement, "history": history}, last_path)
        pd.DataFrame(history).to_csv(run_dir / "results.csv", index=False)

    done_marker.write_text("finished")
    print(f"{run_name(key)}: finished. Best validation mAP50-95 = {best_score:.3f}")

### Train every baseline in the list

In [11]:
if RUN_BASELINES:
    for key in BASELINES:
        print("=" * 70)
        train_baseline(key)
        torch.cuda.empty_cache()   # hand the GPU memory back before the next model
else:
    print("Part 2 switched off.")

detect_fasterrcnn_resnet50_fpn_v2_e30_p30_img640: already finished, skipping.
detect_efficientdet_d0_e30_p30_img640: already finished, skipping.


### Final scores on val and test

The best checkpoint of each model (chosen on validation) is now scored on the test split. As before: look at test once, at the end.

In [12]:
overall_rows, per_class_rows = [], []

for key in (BASELINES if RUN_BASELINES else []):
    run_dir = NEW_RUNS_DIR / run_name(key)
    best_path = run_dir / "weights" / "best.pt"
    if not best_path.exists():
        print(f"Skipping {key}: not trained yet.")
        continue
    wrapper = WRAPPERS[key]()
    wrapper.model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    history = pd.read_csv(run_dir / "results.csv")
    n_params = sum(p.numel() for p in wrapper.model.parameters())

    for split in ("val", "test"):
        overall, per_class = evaluate(wrapper, split)
        overall_rows.append({"model": key, "split": split, "settings": SETTINGS_TAG, **overall,
                             "params_millions": round(n_params / 1e6, 2),
                             "epochs_run": len(history),
                             "best_epoch": int(history.loc[history["mAP50-95"].idxmax(), "epoch"])})
        per_class_rows.extend({"model": key, "split": split, "settings": SETTINGS_TAG, **row} for row in per_class)
    del wrapper
    torch.cuda.empty_cache()

baselines_overall_df = pd.DataFrame(overall_rows)
baselines_per_class_df = pd.DataFrame(per_class_rows)
if not baselines_overall_df.empty:
    baselines_overall_df.to_csv(SUMMARY_DIR / f"baselines_overall_{SETTINGS_TAG}.csv", index=False)
    baselines_per_class_df.to_csv(SUMMARY_DIR / f"baselines_per_class_{SETTINGS_TAG}.csv", index=False)
baselines_overall_df

,model,split,settings,precision,recall,mAP50,mAP50-95,inference_ms_per_image,params_millions,epochs_run,best_epoch
0,fasterrcnn_resnet50_fpn_v2,val,e30_p30_img640,0.856,0.900,0.914,0.669,83.56,43.28,30,27
1,fasterrcnn_resnet50_fpn_v2,test,e30_p30_img640,0.875,0.874,0.898,0.644,85.79,43.28,30,27
2,efficientdet_d0,val,e30_p30_img640,0.875,0.894,0.918,0.668,13.25,3.83,30,17
3,efficientdet_d0,test,e30_p30_img640,0.852,0.883,0.907,0.643,10.03,3.83,30,17


## Part 3: the master table

This gathers every summary table from `runs_v2/summary` (the compare-v2 notebook) and `runs_v3/summary` (this notebook) into one file, `runs_v3/summary/all_models_overall.csv`. The `settings` column tells the experiments apart:

- `april_cpu_e30_img640`: your original April runs
- `e30_p30_img640`: the main YOLO results
- `e100_p20_img640`: the long-schedule experiment
- `e30_p30_img1280`: the image-size experiment
- baselines use `e30_p30_img640` too, and are told apart by the model name

In [13]:
def load_tables(kind: str) -> pd.DataFrame:
    # kind is "overall" or "per_class"
    frames = []
    april = V2_SUMMARY_DIR / f"april_models_{kind}.csv"
    if april.exists():
        df = pd.read_csv(april)
        df["model"] = df["model"].str.split(" ").str[0]     # "yolov8n (April, 30 ep)" -> "yolov8n"
        df["settings"] = "april_cpu_e30_img640"
        frames.append(df)
    for pattern in (f"new_models_{kind}_*.csv", f"baselines_{kind}_*.csv"):
        for f in sorted(V2_SUMMARY_DIR.glob(pattern)) + sorted(SUMMARY_DIR.glob(pattern)):
            df = pd.read_csv(f)
            if "settings" not in df.columns or df["settings"].isna().all():
                df["settings"] = f.stem.split(f"{kind}_", 1)[1]   # older files: read the tag from the file name
            frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No summary tables found in {V2_SUMMARY_DIR} or {SUMMARY_DIR}.")
    return pd.concat(frames, ignore_index=True)

all_overall = load_tables("overall")
all_per_class = load_tables("per_class")
all_overall.to_csv(SUMMARY_DIR / "all_models_overall.csv", index=False)
all_per_class.to_csv(SUMMARY_DIR / "all_models_per_class.csv", index=False)

columns = ["model", "settings", "precision", "recall", "mAP50", "mAP50-95", "inference_ms_per_image", "params_millions"]
print("Test split, all experiments:")
display(all_overall[all_overall["split"] == "test"][columns].sort_values(["settings", "mAP50-95"], ascending=[True, False]))

Test split, all experiments:


,model,settings,precision,recall,mAP50,mAP50-95,inference_ms_per_image,params_millions
1,yolov8n,april_cpu_e30_img640,0.880,0.848,0.921,0.668,3.22,3.01
3,yolo11n,april_cpu_e30_img640,0.877,0.874,0.914,0.668,3.37,2.59
5,yolo26n,april_cpu_e30_img640,0.851,0.860,0.897,0.663,3.58,2.51
9,yolo11n,e100_p20_img640,0.877,0.807,0.904,0.659,3.31,2.61
7,yolov8n,e100_p20_img640,0.885,0.843,0.914,0.651,3.72,3.14
11,yolo26n,e100_p20_img640,0.819,0.886,0.895,0.650,4.21,2.54
23,yolo26n,e30_p30_img1280,0.888,0.867,0.916,0.699,9.71,2.54
21,yolo11n,e30_p30_img1280,0.861,0.882,0.912,0.683,10.25,2.61
19,yolov8n,e30_p30_img1280,0.885,0.858,0.907,0.675,9.84,3.14
13,yolov8n,e30_p30_img640,0.891,0.869,0.923,0.675,3.38,3.14


In [14]:
# Minion check: did the bigger image size help the small classes?
# One row per class, one column per (model, settings) pair. Test split, AP50-95.
test_pc = all_per_class[all_per_class["split"] == "test"].copy()
test_pc["run"] = test_pc["model"] + " | " + test_pc["settings"]
display(test_pc.pivot_table(index="class", columns="run", values="AP50-95"))

run,efficientdet_d0 | e30_p30_img640,fasterrcnn_resnet50_fpn_v2 | e30_p30_img640,yolo11n | april_cpu_e30_img640,yolo11n | e100_p20_img640,yolo11n | e30_p30_img1280,yolo11n | e30_p30_img640,yolo26n | april_cpu_e30_img640,yolo26n | e100_p20_img640,yolo26n | e30_p30_img1280,yolo26n | e30_p30_img640,yolov8n | april_cpu_e30_img640,yolov8n | e100_p20_img640,yolov8n | e30_p30_img1280,yolov8n | e30_p30_img640
class,,,,,,,,,,,,,,
ally_champion,0.709,0.696,0.738,0.728,0.785,0.712,0.716,0.698,0.764,0.738,0.721,0.686,0.738,0.736
ally_minion,0.495,0.491,0.571,0.560,0.557,0.566,0.576,0.562,0.608,0.583,0.562,0.573,0.592,0.559
enemy_champion,0.712,0.729,0.721,0.728,0.745,0.735,0.716,0.700,0.759,0.697,0.729,0.721,0.781,0.736
enemy_minion,0.434,0.442,0.519,0.476,0.507,0.484,0.505,0.495,0.512,0.529,0.508,0.510,0.488,0.523
player,0.803,0.794,0.789,0.794,0.806,0.815,0.798,0.806,0.839,0.801,0.788,0.749,0.805,0.787
tower,0.707,0.711,0.672,0.667,0.698,0.718,0.667,0.637,0.711,0.654,0.699,0.664,0.643,0.711


## 11. Reading the results fairly

Three caveats to write into the report, because an examiner will think of them anyway:

1. **Different training recipes.** YOLO used its built-in tuned recipe with strong augmentation. The baselines used a plain one. If YOLO wins, part of that is the recipe, not only the architecture.
2. **Different sizes.** Faster R-CNN has roughly 43 million parameters, a nano YOLO about 2.5 million. Compare accuracy *and* speed *and* size, not accuracy alone.
3. **Speed numbers are measured differently.** YOLO's speed comes from Ultralytics' own timer, the baselines' from the stopwatch in this notebook. Treat speed as a rough comparison (for example "about five times slower"), not as exact figures.